In [0]:
# Instalar dependencias desde requirements.txt
%pip install -r ../requirements.txt
%pip install langchain langchain-databricks langchain-community
dbutils.library.restartPython()

In [0]:
# Widgets de configuración para endpoints y nombre de índice
# Core Configuration
dbutils.widgets.text("catalog_name", "bluetab", "Catalog Name")
dbutils.widgets.text("schema_name", "rag", "Schema Name")
dbutils.widgets.text("environment", "dev", "Environment (dev/test/prod)")
dbutils.widgets.text("index_name", "bluetab.rag.docs_idx", "Index Name")
dbutils.widgets.text("embedding_endpoint", "simple_embbeding", "Embedding Endpoint")
dbutils.widgets.text("llm_endpoint", "flan_t5_base_model", "LLM Endpoint")
dbutils.widgets.text("vector_search_endpoint", "doc_vector_endpoint", "Vector Search Endpoint")

# MLflow Configuration
dbutils.widgets.text("experiment_name", "/Shared/RAG_Databricks_Bluetab_Pipeline", "MLflow Experiment Name")

# MLflow run management
dbutils.widgets.text("parent_run_id", "", "Parent Run ID")
dbutils.widgets.text("current_run", "", "Current Run")

In [0]:
# Obtener valores de los widgets y definir variables principales
CATALOG_NAME = dbutils.widgets.get("catalog_name")
SCHEMA_NAME = dbutils.widgets.get("schema_name")
ENVIRONMENT = dbutils.widgets.get("environment")

INDEX_NAME = dbutils.widgets.get("index_name")
EMBEDDING_ENDPOINT = dbutils.widgets.get("embedding_endpoint")
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint")
VECTOR_SEARCH_ENDPOINT = dbutils.widgets.get("vector_search_endpoint")

# MLflow configuration
EXPERIMENT_NAME = dbutils.widgets.get("experiment_name")

# Variables globales para gestión de parent/child runs
PARENT_RUN_ID = dbutils.widgets.get("parent_run_id") or None
CURRENT_RUN = dbutils.widgets.get("current_run") or None

print(f"Index Name: {INDEX_NAME}")
print(f"Embedding Endpoint: {EMBEDDING_ENDPOINT}")
print(f"LLM Endpoint: {LLM_ENDPOINT}")
print(f"Vector Search Endpoint: {VECTOR_SEARCH_ENDPOINT}")

In [0]:
%run "./00 Configuration and Utils"

In [0]:
start_child_run("09_create_chatbot")

In [0]:
from langchain.embeddings.base import Embeddings
from typing import List
import mlflow.deployments

class BluetabEmbeddingModel(Embeddings):
    def __init__(self, endpoint_name: str):
        self.endpoint_name = endpoint_name
        self.client = mlflow.deployments.get_deploy_client("databricks")

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._embed(text) for text in texts]

    def embed_query(self, text: str) -> List[float]:
        return self._embed(text)

    def _embed(self, text: str) -> List[float]:
        input_data = {
            "dataframe_split": {
                "columns": ["input"],
                "data": [[text]]
            }
        }
        response = self.client.predict(endpoint=self.endpoint_name, inputs=input_data)

        # Accede directamente al primer embedding de la lista
        return response["predictions"][0]


In [0]:
import os
from langchain_databricks import DatabricksEmbeddings, ChatDatabricks
# Importa DatabricksVectorSearch desde la librería correcta
from langchain_community.vectorstores import DatabricksVectorSearch
from databricks.vector_search.client import VectorSearchClient

from langchain.chains import RetrievalQA

# --- 1. Configuración Segura y Simplificada ---
# Es la mejor práctica configurar las credenciales como variables de entorno.
# LangChain las leerá automáticamente.
# Asegúrate de haber ejecutado estas líneas o tenerlas configuradas en tu entorno.
# os.environ['DATABRICKS_HOST'] = "https://dbc-ad7d5e59-0280.cloud.databricks.com/"
# os.environ['DATABRICKS_TOKEN'] = ""

# Nombres de tus endpoints y tu índice
# INDEX_NAME = "bluetab.rag.docs_idx"
# EMBEDDING_ENDPOINT = "simple_embbeding"
# LLM_ENDPOINT = "flan_t5_base_model"
# VECTOR_SEARCH_ENDPOINT = "doc_vector_endpoint"

# --- 2. Inicialización de Componentes ---

# Modelo de Embeddings (tu código ya era correcto)
embedding_model = BluetabEmbeddingModel(endpoint_name=EMBEDDING_ENDPOINT)

# Vector Store (forma simplificada y correcta)
# No necesitas crear un VectorSearchClient manualmente.
# La clase de LangChain solo necesita el nombre del endpoint del índice.
def get_retriever(persist_dir: str = None):
    #Get the vector search index
    vs_client = VectorSearchClient()
    vs_index = vs_client.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=INDEX_NAME
    )
    vectorstore = DatabricksVectorSearch(
    index=vs_index,
    embedding=embedding_model,
    text_column="text"
    )

    # El retriever se crea a partir del vector store
    return vectorstore.as_retriever()

In [0]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# LLM para la generación de respuestas
llm = ChatDatabricks(endpoint=LLM_ENDPOINT, max_tokens=200)

# --- 3. Creación del Retriever y la Cadena RAG ---

TEMPLATE = """You are an internal assistant chatbot for Bluetab employees. You answer questions related to Bluetab’s company information, products, technologies, employee guidelines, and internal processes. If the question is outside of these topics, politely decline to answer. If you don't know the answer, clearly state that you don't know and avoid inventing answers. If the question is about products or services not related to Bluetab, say so. Keep your answers clear and concise. Provide all answers only in Spanish.

Use the following pieces of context to answer the question at the end:
{context}
Pregunta: {question}
Respuesta:
"""

prompt = PromptTemplate(template=TEMPLATE, input_variables=["context", "question"])

chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=get_retriever(),
    chain_type_kwargs={"prompt": prompt}
)



In [0]:
# --- 4. Ejecución de la Cadena ---

question = "¿Qué es bluetab?"
response = chain.invoke({"query": question})

print(response['result'])

In [0]:
from mlflow.models import infer_signature
import mlflow
import langchain

mlflow.set_registry_uri("databricks-uc")
model_name = "bluetab.rag.appliance_chatbot_model"

signature = infer_signature(question, response)
model_info = mlflow.langchain.log_model(
    chain,
    loader_fn=get_retriever,  # Load the retriever with DATABRICKS_TOKEN env as secret (for authentication).
    artifact_path="chain",
    registered_model_name=model_name,
    pip_requirements=[
        "mlflow==" + mlflow.__version__,
        "langchain==" + langchain.__version__,
        "langchain-community",
        "langchain-databricks",
        "databricks-vectorsearch",
        "databricks-langchain",
    ],
    input_example=question,
    signature=signature
)